# 方案 A 教学入口：从数据到结果，理解每一步

**主题：乳腺肿块良恶性分类中的特征减量。**

这份 Notebook 用于阅读项目已经生成的真实数据、统计表和图形。它不维护第二套建模代码；完整计算统一交给项目根目录的 `run_analysis.py`。

本文件所有代码格初始未执行，未保存任何假造输出。项目中的 `outputs/` 是已实际运行的分析产物，读取它们与重新运行分析是两件事。

学习顺序：找到项目 → 认识原始数据 → 质量和划分 → 可选完整复现 → 统计分析 → 开发集验证 → 测试评价 → 手算指标 → 特征与图形 → 自检。

**先记住：**M=1 表示恶性；ID 不进入模型；开发集用于比较与选择，测试集用于最后评价。不能因为看到了测试结果，再改特征数、随机种子、阈值或默认模型。

## 1. 找到项目根目录

**输入：**当前工作目录 `Path.cwd()`。

**操作：**检查当前目录、父目录及课程中的约定子目录，找到同时含 `run_analysis.py` 和 `config.json` 的位置。将其记作 `PROJECT_ROOT`，之后所有路径都由它构造。

**输出：**实际工作目录与项目根目录。若找不到，会停止并给出明确说明；不要把任意报错简单理解成“代码错了”。

In [ ]:
from pathlib import Path
import json
import sys

working_directory = Path.cwd().resolve()
candidates = []
for base in [working_directory, *working_directory.parents]:
    candidates.extend([
        base,
        base / "方案A项目",
        base / "课程梳理" / "方案A项目",
    ])
PROJECT_ROOT = next(
    (candidate for candidate in candidates
     if (candidate / "run_analysis.py").is_file()
     and (candidate / "config.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "找不到方案A项目。请把工作目录设为方案A项目或其notebooks目录，再运行本格。"
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS = PROJECT_ROOT / "outputs"
config = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))
print("当前工作目录:", working_directory)
print("项目根目录:", PROJECT_ROOT)
print("固定标签映射:", config["label_mapping"])

## 2. 原始 CSV：为什么是 32 列，却只有 30 个输入特征？

**输入：**`data/raw/wdbc.csv` 与项目已经定义好的特征名列表。

**操作：**用 `header=None` 读取无表头文件；前两列分别是 ID 与 B/M 标签，后 30 列是数值特征。复用项目的 `FEATURES`，避免另写一套列名导致顺序不一致。

**输出：**原始形状、少量预览和两类数量。这里是在解释数据格式，不重新选择研究问题或特征。

In [ ]:
import pandas as pd
import numpy as np
from src.analysis import FEATURES

raw_table = pd.read_csv(
    PROJECT_ROOT / config["source_csv"],
    header=None,
    dtype={0: str, 1: str},
)
raw_table.columns = ["id", "diagnosis"] + FEATURES
raw_table["target"] = raw_table["diagnosis"].map({"B": 0, "M": 1})
X_preview = raw_table[FEATURES]
y_preview = raw_table["target"]

print("原始文件行列数（不含新加的target）:", (len(raw_table), len(raw_table.columns) - 1))
print("模型输入的形状:", X_preview.shape)
print("ID是否进入模型:", "id" in X_preview.columns)
print("标签计数:")
print(raw_table["diagnosis"].value_counts().to_string())
print("少量预览:")
print(raw_table[["id", "diagnosis", "target", "mean_radius", "mean_texture"]].head().to_string(index=False))
assert y_preview.notna().all()
assert X_preview.shape == (569, 30)
assert "id" not in X_preview.columns

### 列名和医学含义

30 列由 10 种细胞核特征的 `mean_`、`se_`、`worst_` 三类摘要构成。`worst_` 在原数据说明中指三个最大值的均值，不是病情等级。

这些数值已经从细针穿刺图像提取。减少模型输入列数不等于减少检查次数，也不能据此证明临床诊断安全。本项目研究的是既定数据和验证条件下的模型表现。

## 3. 读取质量检查和固定划分

**输入：**真实运行生成的 `outputs/quality.json` 和 `data/processed/split_assignments.csv`。

**操作：**查看缺失、无穷值、重复 ID 等质量记录；按 development/test 汇总类别数量，确认它们没有重复使用同一条记录。

**输出：**质量记录与分区计数。没有缺失不意味着可以跳过预处理：ID 排除、标签编码、划分和训练内标准化仍属于预处理。

In [ ]:
quality = json.loads((OUTPUTS / "quality.json").read_text(encoding="utf-8"))
split_table = pd.read_csv(
    PROJECT_ROOT / "data/processed/split_assignments.csv", dtype={"id": str}
)
split_counts = pd.crosstab(split_table["partition"], split_table["target"])
split_counts = split_counts.rename(columns={0: "良性B", 1: "恶性M"})
split_counts["总数"] = split_counts.sum(axis=1)
print("质量核查:")
print(json.dumps(quality, ensure_ascii=False, indent=2))
print("固定划分:")
print(split_counts.to_string())
assert split_table["row_index"].is_unique
assert len(split_table) == len(raw_table)
assert set(split_table["partition"]) == {"development", "test"}

## 4. 可选：通过唯一分析入口完整重现

**输入：**现有 `config.json`、原始数据副本及项目运行脚本。

**操作：**默认只说明运行命令。确实准备完整重跑时，将 `RUN_FULL_ANALYSIS` 改成 `True`。脚本会重新生成 `outputs/` 的分析结果；原始课程数据不会被修改。不要为了提高测试分数修改配置。

**输出：**关闭开关时只打印提示；开启时由现有脚本生成日志、表格、模型和图形。本格没有复制任何模型实现。

运行前请确认当前 Kernel 使用项目所需依赖；若缺包，按项目 README 配置环境，不必反复修改分析代码。

In [ ]:
import subprocess

RUN_FULL_ANALYSIS = False  # 只在你明确准备重现整套流程时改为 True
analysis_command = [sys.executable, str(PROJECT_ROOT / "run_analysis.py")]
if RUN_FULL_ANALYSIS:
    subprocess.run(analysis_command, cwd=PROJECT_ROOT, check=True)
else:
    print("本格未重新执行分析；后续将读取项目已有的真实结果。")
    print("完整重现命令:", " ".join(analysis_command))

## 5. 读取结果总览与测试前冻结记录

**输入：**`outputs/summary.json` 与 `outputs/frozen_selection.json`。

**操作：**检查实际运行状态、默认模型以及选择规则。默认模型只来自 LR30/LR10/LR5 的开发集平均 AUC，不按测试成绩更换。

**输出：**默认模型和测试前冻结日志的时间。时间记录用于追踪执行顺序，模型选择原则仍需要看代码和配置。

In [ ]:
summary = json.loads((OUTPUTS / "summary.json").read_text(encoding="utf-8"))
frozen_selection = json.loads((OUTPUTS / "frozen_selection.json").read_text(encoding="utf-8"))
print("结果状态:", summary["status"])
print("默认展示模型:", summary["default_model"])
print("选择规则:", frozen_selection["selection_rule"])
print("测试前冻结时间UTC:", frozen_selection["frozen_at_utc"])
print("最终拟合开始UTC:", summary["provenance"]["final_fit_started_at_utc"])
assert summary["default_model"] == frozen_selection["default_model"]
assert summary["default_model"] in config["default_candidates"]

## 6. 统计分析：分布差异、P 值和效应量分别是什么？

**输入：**开发集三项预设特征的 `group_comparisons.csv`。

**操作：**读取 B/M 两组的中位数和 Q1/Q3；查看 Mann–Whitney U、Holm 校正 P 值及秩二列相关效应量和区间。

**输出：**实际统计表。U 检验不应一概称为中位数检验；效应量为正表示恶性组更倾向于取较高值。三项变量在结果生成前确定，这些检验不负责选择最终模型输入列。

In [ ]:
statistics_table = pd.read_csv(OUTPUTS / "tables/group_comparisons.csv")
stat_columns = [
    "feature", "benign_n", "benign_median", "benign_q1", "benign_q3",
    "malignant_n", "malignant_median", "malignant_q1", "malignant_q3",
    "u_malignant", "p_raw", "p_holm", "rank_biserial",
    "rank_biserial_ci_low", "rank_biserial_ci_high",
]
print(statistics_table[stat_columns].to_string(index=False))
print("区间解释：B/M各组内部独立有放回抽样1000次的百分位95%区间。")
print("预设统计变量:", config["descriptive_features"])

## 7. 开发集验证：为什么不能先筛选一次，再做交叉验证？

**输入：**五折指标表和每折筛选名单。

**操作：**查看各折样本数、AUC，以及不同折如何重新选择变量。标准化和 SelectKBest 在每个折的训练部分重新拟合；验证部分只沿用规则。

**输出：**五折结果和各模型的均值/标准差。五折之间的标准差不是 95% 置信区间，五折也不是五项独立临床研究。

In [ ]:
cv_table = pd.read_csv(OUTPUTS / "tables/cv_fold_metrics.csv")
cv_summary = pd.read_csv(OUTPUTS / "tables/model_cv_summary.csv")
fold_features = pd.read_csv(OUTPUTS / "tables/cv_selected_features.csv")
print("逐折AUC:")
print(cv_table[["model", "fold", "train_n", "validation_n", "auc"]].to_string(index=False))
print("五折汇总:")
print(cv_summary.to_string(index=False))
print("LR5在不同折选出的列:")
for fold, rows in fold_features.loc[fold_features["model"] == "LR5"].groupby("fold"):
    print("第", fold, "折:", rows["feature"].tolist())

## 8. 最终测试：排序能力不等于固定阈值下的识别能力

**输入：**同一独立测试集上五个预设模型的真实评价表。

**操作：**同时查看 AUC、恶性召回率、特异度、精确率和错误计数。分类阈值固定为 0.5，不是根据测试结果挑出的“最佳阈值”。

**输出：**模型测试比较表。若 Dummy 没有预测任何恶性病例，精确率无定义，CSV 读入后显示 NaN；这不是模型数据丢失，网页/JSON用 null 表示该情况。不要把它默认为完美精确率。

**边界：**看测试表是为了报告既定实验，不是重新选默认模型、特征数或随机种子的依据。

In [ ]:
test_table = pd.read_csv(OUTPUTS / "tables/model_test_metrics.csv")
metric_columns = [
    "model", "k", "auc", "auc_ci_low", "auc_ci_high",
    "sensitivity", "specificity", "precision", "f1", "accuracy",
    "tn", "fp", "fn", "tp", "threshold",
]
print(test_table[metric_columns].to_string(index=False))
print("默认展示模型仍为:", summary["default_model"], "，依据开发集CV而非测试集。")
print("AUC区间只反映固定已拟合模型下的测试样本抽样不确定性。")

## 9. 手算一次恶性召回率：漏判是什么意思？

**输入：**测试集真实标签与 LR30 的已保存预测。

**操作：**逐条判断真实恶性且预测良性的记录，得到 FN；真实恶性且预测恶性得到 TP，再计算 TP/(TP+FN)。

**输出：**真实恶性数、识别数、漏判数与召回率；同时检查它是否与汇总表一致。这里不用新的模型，也不修改任何阈值。

In [ ]:
predictions = pd.read_csv(OUTPUTS / "predictions/test_predictions.csv", dtype={"id": str})
example_model = "LR30"
true_label = predictions["true_label"].to_numpy()
predicted_label = predictions[example_model + "_prediction"].to_numpy()
TP = int(((true_label == 1) & (predicted_label == 1)).sum())
FN = int(((true_label == 1) & (predicted_label == 0)).sum())
manual_recall = TP / (TP + FN)
reported_recall = float(test_table.loc[test_table["model"] == example_model, "sensitivity"].iloc[0])
print(f"{example_model}: 真实恶性 {TP + FN} 条，识别 {TP} 条，漏判 {FN} 条。")
print(f"手算召回率 = {TP}/({TP}+{FN}) = {manual_recall:.6f}")
print("与保存指标一致:", np.isclose(manual_recall, reported_recall))
assert np.isclose(manual_recall, reported_recall)

## 10. 观察 AUC 差值：接近为什么不等于等效？

**输入：**LR5−LR30 与 LR10−LR30 的成对 bootstrap 结果。

**操作：**读取点估计及 95% 区间。同一次重采样对所有模型使用同一批测试记录，并在真实 B/M 类别内抽样，保留配对关系。

**输出：**减量模型相对 LR30 的 AUC 差值。区间跨零不能证明模型等效，也没有预设临床非劣效界值；该区间不包含重新划分、训练和筛选的全部不确定性。

In [ ]:
difference_table = pd.read_csv(OUTPUTS / "tables/auc_comparisons.csv")
print(difference_table.to_string(index=False))
print("Bootstrap次数:", config["bootstrap_repeats"])
print("抽样方法:", summary["methods"]["test_bootstrap"])
for row in difference_table.itertuples(index=False):
    crosses_zero = row.ci_low <= 0 <= row.ci_high
    print(row.contrast, "区间是否包含0:", crosses_zero, "；不据此作等效结论。")

## 11. 最终特征名单与入选频次

**输入：**完整开发集最终拟合产生的特征名单，以及五折训练中的入选次数。

**操作：**分别读出最终 LR5/LR10 的列名和五折频次，区分“最终模型实际使用了什么”与“不同折的选择是否一致”。

**输出：**实际特征名单和 0–5 次的频数。频次只是探索性的稳定性描述，不是真实入选概率。对高度相关的变量，某一列未入选也不等于没有医学意义。

In [ ]:
final_features = pd.read_csv(OUTPUTS / "tables/final_selected_features.csv")
frequency_table = pd.read_csv(OUTPUTS / "tables/feature_selection_frequency.csv")
for model_id in ["LR5", "LR10", "LR30"]:
    selected = final_features.loc[final_features["model"] == model_id, "feature"].tolist()
    print(model_id, "最终使用", len(selected), "列:", selected)
print("LR5至少入选过一次的特征:")
print(frequency_table.loc[
    (frequency_table["model"] == "LR5") & (frequency_table["count"] > 0)
].to_string(index=False))

## 12. 查看真实图形

**输入：**`outputs/figures/` 中实际生成的 PNG。

**操作：**用 Matplotlib 读图显示，不依赖 IPython 的 `display`。图像只是已有数值结果的呈现；不在这里重画一套不同参数的分析。

**输出：**开发集变量分布、模型表现和成对差值图。对照源 CSV 解读：图中的 SD 与 CI 意义不同，所有极端观测仍保留在分布图中。

In [ ]:
import matplotlib.pyplot as plt

def show_saved_figure(filename, width=11):
    """显示已生成的真实图形，不重新进行统计分析。"""
    path = OUTPUTS / "figures" / filename
    if not path.is_file():
        raise FileNotFoundError(f"没有找到图形: {path}")
    image = plt.imread(path)
    height = width * image.shape[0] / image.shape[1]
    fig, ax = plt.subplots(figsize=(width, height))
    ax.imshow(image)
    ax.axis("off")
    plt.show()
    plt.close(fig)

show_saved_figure("feature_distributions.png")
show_saved_figure("model_performance.png")
show_saved_figure("auc_differences.png")

## 13. 六个学习自检问题

1. 为什么原始文件有 32 列，但模型只接收 30 列？如果没有指定 `header=None`，会发生什么？
2. 为什么标准化和特征筛选必须放进交叉验证每折的训练过程？在整个开发集先筛一次，再交叉验证，哪里用了不该提前使用的信息？
3. 三项预设变量的统计检验、效应量，以及 SelectKBest 的筛选，分别解决什么问题？
4. 为什么两个模型的 AUC 可以接近，但阈值 0.5 下的恶性召回率不同？用 TP、FN 说明召回率。
5. 为什么“减量模型与 LR30 的 AUC 差区间跨零”不能证明等效？当前区间没有覆盖哪些不确定性？
6. 为什么默认展示模型必须保持测试前的选择？为什么减少输入列数，不能直接推导减少临床检查或降低诊疗成本？

完成以上解释后，再读报告和网页。它们应当使用本项目同一批真实结果；网页的 5/10/30 切换只改变展示，不重新训练或改变实验结论。